In [1]:
import json
import os

from dotenv import load_dotenv

from pydantic import BaseModel, ValidationError, Field
from typing import List

from openai import AsyncOpenAI

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread

from semantic_kernel.functions import KernelArguments

In [2]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("API_KEY"), 
    base_url=os.environ.get("API_URL"),
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id=os.environ.get("MODEL_FREE_8B"),
    async_client=client,
)

In [3]:
class SubTask(BaseModel):
    assigned_agent: str = Field(
        description="负责处理此子任务的特定代理")
    task_details: str = Field(
        description="此子任务需要完成的详细描述")


class TravelPlan(BaseModel):
    main_task: str = Field(
        description="用户的整体旅行请求")
    subtasks: List[SubTask] = Field(
        description="从主任务分解的子任务列表，每个子任务分配给专门的代理")

In [4]:
AGENT_NAME = "TravelAgent"
AGENT_INSTRUCTIONS = """你是一个规划代理。
你的工作是根据用户的请求决定运行哪些代理。
以下是专注于不同任务的可用代理：
- FlightBooking: 用于预订航班和提供航班信息
- HotelBooking: 用于预订酒店和提供酒店信息
- CarRental: 用于预订汽车和提供汽车租赁信息
- ActivitiesBooking: 用于预订活动和提供活动信息
- DestinationInfo: 用于提供关于目的地的信息
- DefaultAgent: 用于处理一般请求"""

# 创建提示执行设置并配置Pydantic模型响应格式
settings = OpenAIChatPromptExecutionSettings(response_format=TravelPlan)

agent = ChatCompletionAgent(
    service=chat_completion_service,
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    arguments=KernelArguments(settings) 
)

In [5]:
from IPython.display import display, HTML


async def main():
    # 创建代理线程
    # 如果没有提供线程，将创建一个新线程并与初始响应一起返回
    thread: ChatHistoryAgentThread | None = None

    # 响应用户输入
    user_inputs = [
        "为一个四口之家（有两个孩子）创建从新加坡到墨尔本的旅行计划",
    ]

    for user_input in user_inputs:
        
        # 开始构建HTML输出
        html_output = "<div style='margin-bottom:10px'>"
        html_output += "<div style='font-weight:bold'>用户:</div>"
        html_output += f"<div style='margin-left:20px'>{user_input}</div>"
        html_output += "</div>"

        # 收集代理的响应
        response = await agent.get_response(messages=user_input, thread=thread)
        thread = response.thread

        try:
            # 尝试将响应验证为TravelPlan
            travel_plan = TravelPlan.model_validate(json.loads(response.message.content))

            # 以格式化JSON显示验证后的模型
            formatted_json = travel_plan.model_dump_json(indent=4)
            html_output += "<div style='margin-bottom:20px'>"
            html_output += "<div style='font-weight:bold'>验证后的旅行计划:</div>"
            html_output += f"<pre style='margin-left:20px; padding:10px; border-radius:5px;'>{formatted_json}</pre>"
            html_output += "</div>"
        except ValidationError as e:
            # 处理验证错误
            html_output += "<div style='margin-bottom:20px; color:red;'>"
            html_output += "<div style='font-weight:bold'>验证错误:</div>"
            html_output += f"<pre style='margin-left:20px;'>{str(e)}</pre>"
            html_output += "</div>"
            # 添加此内容以查看响应包含的内容以进行调试
            html_output += "<div style='margin-bottom:20px;'>"
            html_output += "<div style='font-weight:bold'>原始响应:</div>"
            html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{response.content}</div>"
            html_output += "</div>"

        html_output += "<hr>"

        # 显示格式化的HTML
        display(HTML(html_output))

await main()

你应该会看到类似以下的示例输出：

```json
用户:
为一个四口之家（有两个孩子）创建从新加坡到墨尔本的旅行计划
验证后的旅行计划:
{
    "main_task": "为4人（包括2个孩子）计划从新加坡到墨尔本的家庭旅行。",
    "subtasks": [
        {
            "assigned_agent": "FlightBooking",
            "task_details": "为2名成人和2名儿童预订从新加坡到墨尔本的往返机票。"
        },
        {
            "assigned_agent": "HotelBooking",
            "task_details": "在墨尔本寻找并预订适合4人入住的家庭友好型酒店。"
        },
        {
            "assigned_agent": "CarRental",
            "task_details": "在墨尔本安排适合四口之家的汽车租赁。"
        },
        {
            "assigned_agent": "ActivitiesBooking",
            "task_details": "计划并预订墨尔本适合儿童的家庭友好型活动。"
        },
        {
            "assigned_agent": "DestinationInfo",
            "task_details": "提供关于墨尔本的信息，包括景点、餐饮选择和适合家庭的活动。"
        }
    ]
}
```